# Phase 1 - Hoang Intent-Aware Routing Research

Notebook nay bao gom dung 3 buoc cua Vinh Hoang:
1. Intent Classification
2. Dynamic Retrieval Routing
3. Output to Retrieval Contract (`phase1_hoang_intent_routing_output.jsonl`)


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'aviation_rag').exists() and (ROOT.parent / 'aviation_rag').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from aviation_rag.config import Settings
from aviation_rag.io_utils import write_jsonl
from aviation_rag.phase1_hoang_intent_routing import Phase1HoangIntentRouting

settings = Settings()
print('Project root:', settings.project_root)
print('Data path for research mode:', settings.data_path)
print('Phase 1 artifact:', settings.phase1_output_path)
print('Intent mode:', settings.input_intent_mode)


Project root: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project
Data path for research mode: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\data\kaggle\ASRS-clean-dataset-aviation-safety.csv
Phase 1 artifact: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase1_hoang_intent_routing_output.jsonl
Intent mode: heuristic


## Sample Queries

Mot tap query dai dien cho 4 intent trong full flow.


In [2]:
sample_queries = [
    {'query_id': 'q_incident_001', 'query_raw': 'engine failure after takeoff with emergency return'},
    {'query_id': 'q_procedure_001', 'query_raw': 'den bao ENG OIL PRESS sang thi lam gi?'},
    {'query_id': 'q_metadata_001', 'query_raw': 'crosswind turbulence during final approach at runway 25'},
    {'query_id': 'q_factoid_001', 'query_raw': 'what is the meaning of MEL in aviation?'},
]
sample_queries


[{'query_id': 'q_incident_001',
  'query_raw': 'engine failure after takeoff with emergency return'},
 {'query_id': 'q_procedure_001',
  'query_raw': 'den bao ENG OIL PRESS sang thi lam gi?'},
 {'query_id': 'q_metadata_001',
  'query_raw': 'crosswind turbulence during final approach at runway 25'},
 {'query_id': 'q_factoid_001',
  'query_raw': 'what is the meaning of MEL in aviation?'}]

## Phase 1A - Intent Classification + Query Rewriting


In [3]:
phase1 = Phase1HoangIntentRouting(settings)
phase1_rows = []
for item in sample_queries:
    output = phase1.build_output(
        query_raw=item['query_raw'],
        query_id=item['query_id'],
        top_k=10,
    )
    phase1_rows.append(output)

[
    {
        'query_id': row.query_id,
        'intent': row.intent,
        'intent_source': row.intent_source,
        'intent_confidence': row.intent_confidence,
        'rewritten_query': row.rewritten_query,
    }
    for row in phase1_rows
]


[{'query_id': 'q_incident_001',
  'intent': 'Incident_Report',
  'intent_source': 'heuristic',
  'intent_confidence': 0.6,
  'rewritten_query': 'aviation incident narrative lookup for: engine failure after takeoff with emergency return'},
 {'query_id': 'q_procedure_001',
  'intent': 'Technical_Procedure',
  'intent_source': 'heuristic',
  'intent_confidence': 0.6,
  'rewritten_query': 'aviation troubleshooting and procedure lookup for: den bao engine oil press sang thi lam gi'},
 {'query_id': 'q_metadata_001',
  'intent': 'Metadata_Query',
  'intent_source': 'heuristic',
  'intent_confidence': 0.6,
  'rewritten_query': 'aviation metadata and operating condition lookup for: crosswind turbulence during final approach at runway 25'},
 {'query_id': 'q_factoid_001',
  'intent': 'Factoid',
  'intent_source': 'heuristic',
  'intent_confidence': 0.6,
  'rewritten_query': 'direct aviation fact lookup for: what is the meaning of mel in aviation'}]

## Phase 1B - Dynamic Retrieval Routing


In [4]:
[
    {
        'query_id': row.query_id,
        'intent': row.intent,
        'strategy': row.retrieval_plan.strategy,
        'fallback_strategy': row.retrieval_plan.fallback_strategy,
        'filters': row.retrieval_plan.filters,
        'routing_reason': row.retrieval_plan.routing_reason,
    }
    for row in phase1_rows
]


[{'query_id': 'q_incident_001',
  'intent': 'Incident_Report',
  'strategy': 'semantic',
  'fallback_strategy': 'hybrid',
  'filters': {},
  'routing_reason': 'Narrative incident queries benefit from semantic similarity over safety reports.'},
 {'query_id': 'q_procedure_001',
  'intent': 'Technical_Procedure',
  'strategy': 'bm25',
  'fallback_strategy': 'hybrid',
  'filters': {'document_type': 'procedure'},
  'routing_reason': 'Procedure-style queries favor keyword-heavy checklist and manual retrieval.'},
 {'query_id': 'q_metadata_001',
  'intent': 'Metadata_Query',
  'strategy': 'metadata_first',
  'fallback_strategy': 'bm25',
  'filters': {'prefer_metadata': True},
  'routing_reason': 'Metadata queries should prioritize structured document fields and filters first.'},
 {'query_id': 'q_factoid_001',
  'intent': 'Factoid',
  'strategy': 'semantic',
  'fallback_strategy': 'hybrid',
  'filters': {'answer_style': 'short'},
  'routing_reason': 'Factoid queries need concise semantic lookup

## Phase 1C - Export Contract for San


In [5]:
write_jsonl(settings.phase1_output_path, phase1_rows)
required = [
    'query_id', 'query_raw', 'query_normalized', 'intent', 'intent_confidence',
    'intent_source', 'expanded_queries', 'rewritten_query', 'retrieval_plan', 'created_at'
]
for row in phase1_rows:
    payload = row.model_dump(mode='json')
    missing = [key for key in required if key not in payload]
    if missing:
        raise ValueError(f'Missing keys in {payload.get("query_id")}: {missing}')
print(f'Wrote {len(phase1_rows)} rows to {settings.phase1_output_path}')
print('Phase 1 contract validation passed.')


Wrote 4 rows to C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase1_hoang_intent_routing_output.jsonl
Phase 1 contract validation passed.


## Phase 1D - Academic Evaluation Metrics

This cell turns Phase 1 from a demo into a measurable research component. The goal is to show whether the intent router chooses the expected label and retrieval strategy for representative aviation queries.

Definitions:

- Intent accuracy: correct predicted intent divided by total labeled sample queries.
- Routing accuracy: correct retrieval strategy divided by total labeled sample queries.
- Confidence summary: min/mean/max confidence across test queries.


In [6]:
import json
from statistics import mean

expected = {
    'q_incident_001': {'intent': 'Incident_Report', 'strategy': 'semantic'},
    'q_procedure_001': {'intent': 'Technical_Procedure', 'strategy': 'bm25'},
    'q_metadata_001': {'intent': 'Metadata_Query', 'strategy': 'metadata_first'},
    'q_factoid_001': {'intent': 'Factoid', 'strategy': 'semantic'},
}

evaluation_rows = []
for row in phase1_rows:
    gold = expected.get(row.query_id, {})
    evaluation_rows.append({
        'query_id': row.query_id,
        'query_raw': row.query_raw,
        'predicted_intent': row.intent,
        'expected_intent': gold.get('intent'),
        'intent_correct': row.intent == gold.get('intent'),
        'predicted_strategy': row.retrieval_plan.strategy,
        'expected_strategy': gold.get('strategy'),
        'strategy_correct': row.retrieval_plan.strategy == gold.get('strategy'),
        'confidence': round(float(row.intent_confidence), 4),
    })

summary = {
    'sample_size': len(evaluation_rows),
    'intent_accuracy': round(sum(r['intent_correct'] for r in evaluation_rows) / max(1, len(evaluation_rows)), 4),
    'routing_accuracy': round(sum(r['strategy_correct'] for r in evaluation_rows) / max(1, len(evaluation_rows)), 4),
    'confidence_min': round(min(r['confidence'] for r in evaluation_rows), 4),
    'confidence_mean': round(mean(r['confidence'] for r in evaluation_rows), 4),
    'confidence_max': round(max(r['confidence'] for r in evaluation_rows), 4),
}

print(json.dumps({'summary': summary, 'rows': evaluation_rows}, indent=2, ensure_ascii=False))


{
  "summary": {
    "sample_size": 4,
    "intent_accuracy": 1.0,
    "routing_accuracy": 1.0,
    "confidence_min": 0.6,
    "confidence_mean": 0.6,
    "confidence_max": 0.6
  },
  "rows": [
    {
      "query_id": "q_incident_001",
      "query_raw": "engine failure after takeoff with emergency return",
      "predicted_intent": "Incident_Report",
      "expected_intent": "Incident_Report",
      "intent_correct": true,
      "predicted_strategy": "semantic",
      "expected_strategy": "semantic",
      "strategy_correct": true,
      "confidence": 0.6
    },
    {
      "query_id": "q_procedure_001",
      "query_raw": "den bao ENG OIL PRESS sang thi lam gi?",
      "predicted_intent": "Technical_Procedure",
      "expected_intent": "Technical_Procedure",
      "intent_correct": true,
      "predicted_strategy": "bm25",
      "expected_strategy": "bm25",
      "strategy_correct": true,
      "confidence": 0.6
    },
    {
      "query_id": "q_metadata_001",
      "query_raw": "cro